# **PREPARE PROCESSING - CLEANING - SPLITTING**

In [11]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import joblib
from io import StringIO

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split


import warnings

warnings.filterwarnings("ignore")

In [12]:
df = pd.read_csv('diabetes_prediction_dataset.csv')

In [13]:
df_copy = df.copy()

### Xử lý các dữ liệu null/nan, miss, duplicate

In [14]:
# 1. Đếm số dòng trước khi xóa
n_before = df_copy.shape[0]

# 2. Thực hiện lệnh xóa trùng
df_clean = df_copy.drop_duplicates(keep='first')

# 3. Đếm số dòng sau khi xóa
n_after = df_clean.shape[0]
n_duplicates = n_before - n_after

print(f"Số dòng ban đầu: {n_before}")
print(f"Số dòng sau khi xóa trùng: {n_after}")
print(f"Số lượng dòng trùng lặp bị loại bỏ: {n_duplicates}")
print(f"Tỷ lệ mất dữ liệu: {(n_duplicates/n_before)*100:.2f}%")

Số dòng ban đầu: 100000
Số dòng sau khi xóa trùng: 96146
Số lượng dòng trùng lặp bị loại bỏ: 3854
Tỷ lệ mất dữ liệu: 3.85%


In [15]:
#df_copy = df_copy.drop_duplicates(keep='first')
#df_copy.head(5)

In [16]:
# Hiển thị các dòng bị trùng (hiển thị tất cả các bản sao để so sánh)
duplicate_rows = df_copy[df_copy.duplicated(keep=False)]

# Sắp xếp để các dòng giống nhau nằm cạnh nhau cho dễ nhìn
print("Ví dụ về các dòng dữ liệu bị trùng:")
display(duplicate_rows.sort_values(by=list(df_copy.columns)).head(10))

# Kiểm tra xem các dòng trùng này thuộc lớp nào (Tiểu đường hay Không?)
print("\nPhân bố lớp (Label) trong các dòng bị trùng:")
print(duplicate_rows['diabetes'].value_counts())

Ví dụ về các dòng dữ liệu bị trùng:


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
47708,Female,2.0,0,0,No Info,27.32,5.0,158,0
59468,Female,2.0,0,0,No Info,27.32,5.0,158,0
62073,Female,2.0,0,0,No Info,27.32,6.0,85,0
67439,Female,2.0,0,0,No Info,27.32,6.0,85,0
23617,Female,2.0,0,0,No Info,27.32,6.0,145,0
67234,Female,2.0,0,0,No Info,27.32,6.0,145,0
90685,Female,2.0,0,0,No Info,27.32,6.2,145,0
97294,Female,2.0,0,0,No Info,27.32,6.2,145,0
46785,Female,2.0,0,0,No Info,27.32,6.5,155,0
89701,Female,2.0,0,0,No Info,27.32,6.5,155,0



Phân bố lớp (Label) trong các dòng bị trùng:
diabetes
0    6904
1      35
Name: count, dtype: int64


### Mã hóa các đặc trưng Object

In [17]:
#Mã hóa đặc trưng
le = LabelEncoder()
for col in df_copy.select_dtypes(include=['object']).columns:
    df_copy[col] = le.fit_transform(df_copy[col])

In [18]:
df_copy.head(5)

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,0,80.0,0,1,4,25.19,6.6,140,0
1,0,54.0,0,0,0,27.32,6.6,80,0
2,1,28.0,0,0,4,27.32,5.7,158,0
3,0,36.0,0,0,1,23.45,5.0,155,0
4,1,76.0,1,1,1,20.14,4.8,155,0


### Chia thành tập huấn luyện và tập thử nghiệm

In [19]:

# Tách X và y
X = df_copy.drop(columns=['diabetes'])
y = df_copy['diabetes']

# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [20]:

scaler = RobustScaler()

X_train_scaled  = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
print(X_train_scaled )

X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
print(X_test_scaled )

       gender       age  hypertension  heart_disease  smoking_history  \
0         0.0  0.361111           0.0            1.0             0.25   
1         1.0  0.583333           0.0            0.0             0.00   
2         1.0  0.166667           0.0            0.0             0.25   
3         1.0  0.694444           0.0            0.0            -0.75   
4         1.0  0.527778           0.0            0.0             0.50   
...       ...       ...           ...            ...              ...   
69995     0.0 -0.111111           0.0            0.0             0.25   
69996     1.0  0.000000           0.0            0.0             0.25   
69997     0.0  0.250000           1.0            0.0             0.00   
69998     0.0  0.444444           0.0            0.0            -0.50   
69999     1.0 -1.083333           0.0            0.0            -0.75   

            bmi  HbA1c_level  blood_glucose_level  
0      2.055649    -1.285714            -0.677966  
1      0.000000    


       gender       age  hypertension  heart_disease  smoking_history  \
0         1.0 -0.138889           0.0            0.0             0.25   
1         0.0 -0.111111           0.0            0.0             0.50   
2         0.0 -0.250000           0.0            0.0             0.50   
3         0.0  0.000000           0.0            0.0             0.25   
4         0.0  0.527778           0.0            0.0             0.25   
...       ...       ...           ...            ...              ...   
29995     0.0 -1.111111           0.0            0.0            -0.75   
29996     0.0  0.444444           0.0            0.0            -0.50   
29997     0.0 -0.250000           0.0            0.0             0.25   
29998     0.0 -1.083333           0.0            0.0            -0.75   
29999     1.0  0.638889           0.0            0.0            -0.75   

            bmi  HbA1c_level  blood_glucose_level  
0      0.052277    -1.285714             0.084746  
1      1.403035   

In [21]:

# --- Lưu trữ theo cấu trúc của bạn ---
save_dir = "Data_clean"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
    print(f"Đã tạo thư mục '{save_dir}'")

try:
    # 1. Lưu các tập train/validation để huấn luyện và đánh giá
    np.savez(f'{save_dir}/model_data.npz', 
             X_train=X_train_scaled, X_test=X_test_scaled, 
             y_train=y_train, y_test=y_test)
    print(f"- Đã lưu model_data.npz (chứa X_train, X_validation...) vào '{save_dir}'")

    # 2. Lưu các file CSV đã được làm sạch hoàn chỉnh
    df_copy.to_csv(f'{save_dir}/data_cleaned.csv', index=False)
    print(f"- Đã lưu train_cleaned.csv, test_cleaned.csv vào '{save_dir}'")

    # 3. Lưu bộ xử lý (imputer) đã được huấn luyện
    #joblib.dump(fitted_imputer, f'{save_dir}/imputer.joblib')
    print(f"- Đã lưu imputer.joblib vào '{save_dir}'")

    print("\nNội dung thư mục 'Data_clean':")
    print(os.listdir(save_dir))

except Exception as e:
    print(f"\nLỗi khi lưu file: {e}")

- Đã lưu model_data.npz (chứa X_train, X_validation...) vào 'Data_clean'
- Đã lưu train_cleaned.csv, test_cleaned.csv vào 'Data_clean'
- Đã lưu imputer.joblib vào 'Data_clean'

Nội dung thư mục 'Data_clean':
['data_cleaned.csv', 'model_data.npz']
